# LangGraph Agent Workflow 

### _Build Agent using LangGraph:_

##### _# Refer the LangGraph components jupyter notebook for more detailed notes._

Import langgraph state graph, end node , tools, langchain messages, model, typing and operator:

    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated
    import operator
    from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
    from langchain_openai import ChatOpenAI
    from langchain_community.tools.tavily_search import TavilySearchResults

Initialise Tavily search tool with max results as 2 or 4:

    tool = TavilySearchResults(max_results=4) #increased number of results

Create agent state with an annotated list of messages to add to over time:

- Agent state is available at every node and edge of the graph to access the messages

    class AgentState(TypedDict):
        messages: Annotated[list[AnyMessage], operator.add]

Create agent class with 3 functions:

1. Function to call LLM
2. Function to check if action is present
3. Function to perform the action  

=> Build graph in the init function by intialising stategraph, adding nodes, adding conditional edges, adding edges

=> Compile graph and set entry point

=> Specify tools and bind them to the LLM model to use

    class Agent:

        def __init__(self, model, tools, system=""):
            self.system = system
            graph = StateGraph(AgentState)
            graph.add_node("llm", self.call_openai)
            graph.add_node("action", self.take_action)
            graph.add_conditional_edges(
                "llm",
                self.exists_action,
                {True: "action", False: END}
            )
            graph.add_edge("action", "llm")
            graph.set_entry_point("llm")
            self.graph = graph.compile()
            self.tools = {t.name: t for t in tools}
            self.model = model.bind_tools(tools)

        def exists_action(self, state: AgentState):
            result = state['messages'][-1]
            return len(result.tool_calls) > 0

        def call_openai(self, state: AgentState):
            messages = state['messages']
            if self.system:
                messages = [SystemMessage(content=self.system)] + messages
            message = self.model.invoke(messages)
            return {'messages': [message]}

        def take_action(self, state: AgentState):
            tool_calls = state['messages'][-1].tool_calls
            results = []
            for t in tool_calls:
                print(f"Calling: {t}")
                if not t['name'] in self.tools:      # check for bad tool name from LLM
                    print("\n ....bad tool name....")
                    result = "bad tool name, retry"  # instruct LLM to retry if bad
                else:
                    result = self.tools[t['name']].invoke(t['args'])
                results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
            print("Back to the model!")
            return {'messages': results}

Create system prompt to search and find information about a company name:

    prompt = """You are a smart research assistant. Use the search engine tool to find relevant and recent news about the provided company name delimited by ####. Use the information found to format the answer under the title "Company Health Note:" \
    If you cannot find any information about the specific company name, then return 'I could not find any recent information about the company.'
    """

Intialise and call agent with model, tool and system prompt:

    model = ChatOpenAI(model="gpt-3.5-turbo")  
    abot = Agent(model, [tool], system=prompt)

User message asks query to agent and the graph is invoked with the message:

    messages = [HumanMessage(content="####Skillrev####")]
    result = abot.graph.invoke({"messages": messages})

=> Every time user enters query into agent chatbot, the agent graph invoke or runs from starting point  i.e. the LLM model with the user query message

If User provides fake company name to agent and the agent will not hallucinate and return that they couldn't find any information about the company:

    messages = [HumanMessage(content="####Glory Brain Tech Agency####")]
    result = abot.graph.invoke({"messages": messages})

To view AI final response to the user query:

    result['messages'][-1].content